# Bước 1: Khám phá & Chuẩn bị dữ liệu (EDA & Preprocessing)

**Case study Chương 2** — Học Máy Y tế (ET4248) — Đề tài 3.4 "Suy tim cấp tốc"

> Notebook này là **lời giải minh họa đầy đủ** áp dụng lý thuyết Chương 2
> (Mục 2.3 EDA, 2.4 Nền tảng toán, 2.5 Lý thuyết Rubin về dữ liệu thiếu,
> 2.6 Pipeline 4 bước, 2.7 Chuẩn dữ liệu y tế) vào bài toán thật:
> [Heart Failure Clinical Records](https://archive.ics.uci.edu/dataset/519/heart+failure+clinical+records)
> (UCI #519, CC BY 4.0, 299 bệnh nhân, 13 đặc trưng).
>
> ⚠️ **Nếu bạn đang làm Đề tài 3.4 (đồ án được chấm điểm):** notebook này
> là ví dụ minh họa của giáo trình, KHÔNG phải bài nộp hợp lệ cho đồ án —
> xem cảnh báo Zero-Code Policy tại trang
> [Đề tài 3.4](/du_an/suy_tim_risk_dxai) trước khi dùng.

In [1]:
import sys
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler

pd.set_option('display.width', 120)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def _load_heart_failure_data():
    """Tải dữ liệu cục bộ (../data/...) nếu có (chạy trong repo HMYT đã
    clone); nếu không (mở độc lập qua Colab/Kaggle, không có thư mục data/
    đi kèm) tự động tải từ mirror công khai trên hmyt-book (repo Public,
    xác minh 23/09/2026)."""
    import os
    local_path = "../data/heart_failure_clinical_records_dataset.csv"
    remote_url = ("https://raw.githubusercontent.com/fossbk-spec/hmyt-book/gh-pages/"
                  "labs_chuyen_de/ch02_suy_tim_risk_dxai/data/"
                  "heart_failure_clinical_records_dataset.csv")
    path = local_path if os.path.exists(local_path) else remote_url
    if path == remote_url:
        print("[i] Không tìm thấy dữ liệu cục bộ — tự động tải từ mirror công khai: " + remote_url)
    return pd.read_csv(path).rename(columns={"death_event": "DEATH_EVENT"})


## 1. Tải & tổng quan dữ liệu thật (Mục 2.3 — EDA)

Dữ liệu tải trực tiếp từ UCI Machine Learning Repository (không mô phỏng —
khác với Code Lab tổng quát của Chương 2 vốn dùng dữ liệu giả lập).

In [2]:
df = _load_heart_failure_data()

print(f"Số bệnh nhân: {df.shape[0]}  |  Số đặc trưng (không tính nhãn): {df.shape[1] - 1}")
df.head()

Số bệnh nhân: 299  |  Số đặc trưng (không tính nhãn): 12


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       299 non-null    float64
 1   anaemia                   299 non-null    int64  
 2   creatinine_phosphokinase  299 non-null    int64  
 3   diabetes                  299 non-null    int64  
 4   ejection_fraction         299 non-null    int64  
 5   high_blood_pressure       299 non-null    int64  
 6   platelets                 299 non-null    float64
 7   serum_creatinine          299 non-null    float64
 8   serum_sodium              299 non-null    int64  
 9   sex                       299 non-null    int64  
 10  smoking                   299 non-null    int64  
 11  time                      299 non-null    int64  
 12  DEATH_EVENT               299 non-null    int64  
dtypes: float64(3), int64(10)
memory usage: 30.5 KB


**Nhận xét:** tất cả 13 cột đều là kiểu số (`int64`/`float64`), không có
cột dạng chuỗi cần mã hóa — khác với Code Lab tổng quát của Chương 2 (vốn có
cột `gender` dạng phân loại cần `OneHotEncoder`). Đây là điểm khác biệt thật
giữa 2 bài toán, không phải sai sót.

## 2. Kiểm tra dữ liệu khuyết thiếu (Mục 2.5 — Lý thuyết Rubin)

In [4]:
missing = df.isnull().sum()
print("Số giá trị khuyết thiếu theo từng cột:")
print(missing.to_string())
print(f"\nTổng cộng: {missing.sum()} giá trị khuyết thiếu trên {df.size} ô dữ liệu.")

Số giá trị khuyết thiếu theo từng cột:
age                         0
anaemia                     0
creatinine_phosphokinase    0
diabetes                    0
ejection_fraction           0
high_blood_pressure         0
platelets                   0
serum_creatinine            0
serum_sodium                0
sex                         0
smoking                     0
time                        0
DEATH_EVENT                 0

Tổng cộng: 0 giá trị khuyết thiếu trên 3887 ô dữ liệu.


**Kết quả THẬT (không phải mô phỏng):** bộ dữ liệu Heart Failure Clinical
Records **không có bất kỳ giá trị khuyết thiếu nào** — đây là đặc điểm thật
của bộ dữ liệu này (đã được nhóm Chicco & Jurman làm sạch trước khi công bố),
khác với dữ liệu EHR thô trong Code Lab tổng quát của Chương 2.

Vì lý thuyết Rubin (MCAR/MAR/MNAR — Mục 2.5) không có gì để minh họa trực
tiếp trên bộ dữ liệu gốc, phần dưới đây tạo **1 bản sao có khuyết thiếu nhân
tạo** (rõ ràng đây là minh họa bổ sung, KHÔNG phải đặc điểm thật của dữ liệu
gốc) để thực hành đúng 2 cơ chế khuyết thiếu phổ biến trong lâm sàng, theo
đúng khung 4 bước Pipeline (Mục 2.6).

In [5]:
df_demo = df.copy()

# Minh họa MCAR (Missing Completely At Random): lỗi thiết bị đo ngẫu nhiên
# làm mất 8% giá trị serum_sodium, không phụ thuộc vào bất kỳ biến nào khác.
rng = np.random.default_rng(RANDOM_STATE)
mcar_mask = rng.random(len(df_demo)) < 0.08
df_demo.loc[mcar_mask, 'serum_sodium'] = np.nan

# Minh họa MAR (Missing At Random): bác sĩ chỉ định đo creatinine_phosphokinase
# (men tim) phụ thuộc vào tuổi và ejection_fraction quan sát được — bệnh nhân
# trẻ, ejection_fraction bình thường ít được chỉ định đo hơn.
mar_prob = 1 / (1 + np.exp(-(0.04 * (df_demo['age'] - 60) - 0.05 * (df_demo['ejection_fraction'] - 38))))
mar_mask = rng.random(len(df_demo)) > mar_prob
df_demo.loc[mar_mask, 'creatinine_phosphokinase'] = np.nan

print("Tỷ lệ khuyết thiếu nhân tạo (chỉ dùng để minh họa, không phải dữ liệu gốc):")
print((df_demo[['serum_sodium', 'creatinine_phosphokinase']].isnull().mean() * 100).round(2).astype(str) + ' %')

Tỷ lệ khuyết thiếu nhân tạo (chỉ dùng để minh họa, không phải dữ liệu gốc):
serum_sodium                7.02 %
creatinine_phosphokinase    50.5 %
dtype: str


## 3. Phân tích mất cân bằng nhãn (Mục 2.3 — EDA)

In [6]:
counts = df['DEATH_EVENT'].value_counts().sort_index()
ratios = df['DEATH_EVENT'].value_counts(normalize=True).sort_index()
print("Phân bố nhãn DEATH_EVENT (0 = Sống sót, 1 = Tử vong):")
for label, cnt, ratio in zip(counts.index, counts.values, ratios.values):
    name = 'Sống sót' if label == 0 else 'Tử vong'
    print(f"  {label} ({name}): {cnt} bệnh nhân ({ratio*100:.1f}%)")

Phân bố nhãn DEATH_EVENT (0 = Sống sót, 1 = Tử vong):
  0 (Sống sót): 203 bệnh nhân (67.9%)
  1 (Tử vong): 96 bệnh nhân (32.1%)


**Kết quả THẬT:** 203 sống sót (67,9%) / 96 tử vong (32,1%) — mất cân bằng
vừa phải (tỷ lệ ~2,1:1), đúng như mô tả trong bài báo gốc `chicco2020machine`.
Đây là lý do Bước 3 cần áp dụng SMOTE.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
counts.rename({0: 'Sống sót', 1: 'Tử vong'}).plot(kind='bar', ax=axes[0], color=['#4C72B0', '#C44E52'])
axes[0].set_title('Phân bố nhãn DEATH_EVENT (N=299)')
axes[0].set_ylabel('Số bệnh nhân')
axes[0].tick_params(axis='x', rotation=0)

corr = df.corr(numeric_only=True)
sns.heatmap(corr, ax=axes[1], cmap='coolwarm', center=0, annot=False, cbar_kws={'shrink': 0.8})
axes[1].set_title('Ma trận tương quan 13 đặc trưng')

plt.tight_layout()
plt.savefig('../figures/01_eda_overview.png', dpi=110, bbox_inches='tight')
plt.show()
print("[+] Đã lưu ../figures/01_eda_overview.png")

[+] Đã lưu ../figures/01_eda_overview.png


C:\Users\VICTUS\AppData\Local\Temp\ipykernel_23812\2872173697.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
print("Top 5 đặc trưng tương quan mạnh nhất (trị tuyệt đối) với DEATH_EVENT:")
print(corr['DEATH_EVENT'].drop('DEATH_EVENT').abs().sort_values(ascending=False).head(5).round(3).to_string())

Top 5 đặc trưng tương quan mạnh nhất (trị tuyệt đối) với DEATH_EVENT:
time                 0.527
serum_creatinine     0.294
ejection_fraction    0.269
age                  0.254
serum_sodium         0.195


## 4. Phân chia Train/Test chống rò rỉ (Mục 2.6 — Pipeline 4 bước, Bước 1)

*Câu hỏi tư duy (từ đề bài gốc):* Tại sao bạn chỉ được gọi `fit` trên tập
Train mà không được gọi trên tập Test?
**Trả lời:** nếu `fit` (tính mean/std cho `StandardScaler`, hay `fit` bất kỳ
bước tiền xử lý nào) trên toàn bộ dữ liệu (gồm cả Test), thông tin thống kê
của tập Test sẽ "rò rỉ" vào quá trình chuẩn hóa của tập Train — mô hình gián
tiếp "nhìn thấy" phân bố của dữ liệu nó sẽ được đánh giá trên đó, làm chỉ số
hiệu năng đo được lạc quan giả tạo (không phản ánh đúng khả năng tổng quát
hóa trên bệnh nhân mới hoàn toàn).

In [9]:
FEATURE_COLS = [c for c in df.columns if c != 'DEATH_EVENT']
X = df[FEATURE_COLS]
y = df['DEATH_EVENT'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape[0]} bệnh nhân | Test: {X_test.shape[0]} bệnh nhân")
print(f"Tỷ lệ tử vong Train: {y_train.mean()*100:.1f}%  |  Tỷ lệ tử vong Test: {y_test.mean()*100:.1f}%")
print("(2 tỷ lệ trên gần bằng nhau nhờ stratify=y — giữ đúng phân bố nhãn gốc trên cả 2 tập)")

Train: 239 bệnh nhân | Test: 60 bệnh nhân
Tỷ lệ tử vong Train: 32.2%  |  Tỷ lệ tử vong Test: 31.7%
(2 tỷ lệ trên gần bằng nhau nhờ stratify=y — giữ đúng phân bố nhãn gốc trên cả 2 tập)


## 5. Chuẩn hóa đặc trưng — `fit` chỉ trên Train (Mục 2.4 — Nền tảng toán)

In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform trên Train
X_test_scaled = scaler.transform(X_test)          # CHỈ transform trên Test (dùng mean/std đã học từ Train)

print("Mean/Std học được từ Train (5 đặc trưng đầu):")
for name, m, s in list(zip(FEATURE_COLS, scaler.mean_, scaler.scale_))[:5]:
    print(f"  {name:28s}  mean={m:10.3f}  std={s:10.3f}")

Mean/Std học được từ Train (5 đặc trưng đầu):
  age                           mean=    61.073  std=    11.420
  anaemia                       mean=     0.448  std=     0.497
  creatinine_phosphokinase      mean=   602.791  std=  1010.243
  diabetes                      mean=     0.448  std=     0.497
  ejection_fraction             mean=    37.887  std=    11.970


## 6. Đóng gói Pipeline hoàn chỉnh chống rò rỉ (Mục 2.6 — minh họa trên bản demo có khuyết thiếu)

Áp dụng đúng khung `ColumnTransformer` + `Pipeline` như Code Lab tổng quát
của Chương 2, nhưng lần này trên `df_demo` (bản có khuyết thiếu nhân tạo ở
Mục 2) để thực hành cả bước điền khuyết (Imputation) lẫn chuẩn hóa trong
cùng 1 pipeline `fit` duy nhất trên Train.

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

X_demo = df_demo[FEATURE_COLS]
y_demo = df_demo['DEATH_EVENT'].values
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_demo, y_demo, test_size=0.20, stratify=y_demo, random_state=RANDOM_STATE
)

# 2 cột có khuyết thiếu nhân tạo cần Imputer riêng; các cột còn lại chuẩn hóa trực tiếp
missing_cols = ['serum_sodium', 'creatinine_phosphokinase']
clean_cols = [c for c in FEATURE_COLS if c not in missing_cols]

missing_pipeline = Pipeline(steps=[
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler', StandardScaler()),
])
clean_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
])
preprocessor = ColumnTransformer(transformers=[
    ('missing', missing_pipeline, missing_cols),
    ('clean', clean_pipeline, clean_cols),
])
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)),
])

full_pipeline.fit(Xd_train, yd_train)   # 1 lần fit DUY NHẤT trên Train — Imputer + Scaler + Model
yd_prob = full_pipeline.predict_proba(Xd_test)[:, 1]
yd_pred = full_pipeline.predict(Xd_test)

print(f"ROC-AUC (trên bản demo có khuyết thiếu, sau khi điền bằng KNNImputer): {roc_auc_score(yd_test, yd_prob):.4f}")
print(classification_report(yd_test, yd_pred, target_names=['Sống sót', 'Tử vong']))

ROC-AUC (trên bản demo có khuyết thiếu, sau khi điền bằng KNNImputer): 0.8588
              precision    recall  f1-score   support

    Sống sót       0.84      0.93      0.88        41
     Tử vong       0.80      0.63      0.71        19

    accuracy                           0.83        60
   macro avg       0.82      0.78      0.79        60
weighted avg       0.83      0.83      0.83        60



## 7. Tổng kết Bước 1

- Dữ liệu thật **không khuyết thiếu**, nhất quán với việc bài báo gốc
  `chicco2020machine` dùng nguyên 13 đặc trưng không cần điền khuyết.
- Mất cân bằng nhãn thật **32,1% tử vong** — cần xử lý ở Bước 3 (SMOTE).
- Đã dựng xong `X_train`/`X_test`/`y_train`/`y_test` chuẩn (không rò rỉ,
  `stratify=True`, `random_state=42`) — Bước 2 sẽ tái sử dụng đúng tập này.
- Minh họa thêm 1 pipeline chống rò rỉ hoàn chỉnh (Imputer + Scaler + Model
  trong cùng 1 `fit`) trên dữ liệu có khuyết thiếu nhân tạo, đúng khung lý
  thuyết Rubin của Mục 2.5.

**Tiếp theo:** [`2_baseline.ipynb`](./2_baseline.ipynb) — Bước 2, tái lập
kết quả `chicco2020machine` (Accuracy 74,0%, MCC 0,384).